In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Change only the username/password if your MySQL setup is different.
username = 'root'
password = 'YOUR_MYSQL_PASSWORD'
host = 'localhost'
port = 3306
database = 'bingeplay'

engine = create_engine(
    f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
)

print('Connected to:', database)

## Q1 — Project Question

**Answer: 2,340 active subscriptions**  
**Total monthly revenue: ₹784,260**

In [ ]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q2 — Project Question

**Answer:** January 350, February 400, March 500, April 550, May 600, June 600 signups.  
**Highest signup months:** May and June, tied at 600 signups each.

In [ ]:
query = """
SELECT
    MONTH(signup_date) AS month_number,
    DATE_FORMAT(signup_date, '%M') AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY MONTH(signup_date), DATE_FORMAT(signup_date, '%M')
ORDER BY month_number;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

query_highest = """
SELECT
    DATE_FORMAT(signup_date, '%M') AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY MONTH(signup_date), DATE_FORMAT(signup_date, '%M')
ORDER BY signup_count DESC
LIMIT 1;
"""

highest_month = pd.read_sql(query_highest, engine)
print("\nHighest signup month:")
print(highest_month.to_string(index=False))

## Q3 — Project Question

**Answer:** The code prints session count, total watch minutes, average watch minutes, and completion rate for each device. NULL `user_id` sessions are excluded as required.

In [ ]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q4 — Project Question

**Answer:** 1★ = 4.68%, 2★ = 7.04%, 3★ = 16.94%, 4★ = 35.62%, 5★ = 35.72%.  
**4 or 5 stars:** 71.34% of all ratings.

In [ ]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

query_4_5 = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS percentage_4_or_5_stars
FROM ratings;
"""

result_4_5 = pd.read_sql(query_4_5, engine)
print("\nPercentage of ratings that are 4 or 5 stars:")
print(result_4_5.to_string(index=False))

## Q5 — Project Question

**Answer:** Originals: 30 shows, average IMDb 7.92, average release year 2020.37. Acquired: 70 shows, average IMDb 6.63, average release year 2020.73.  
**Interpretation:** In this dataset, Originals have an average IMDb rating 1.29 points higher than acquired content.

In [ ]:
query = """
SELECT
    CASE
        WHEN is_original = 1 THEN 'BingePlay Original'
        ELSE 'Acquired Content'
    END AS content_group,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
    ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q6 — Project Question

**Answer: 414 binge days in Q2 2024.**  
**Top user:** U02956 with 8 binge days.

In [ ]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date,
        COUNT(*) AS session_count
    FROM watch_sessions
    WHERE session_date BETWEEN '2024-04-01' AND '2024-06-30'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
user_binge_counts AS (
    SELECT
        user_id,
        COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
),
top_user AS (
    SELECT
        user_id,
        binge_days,
        ROW_NUMBER() OVER (ORDER BY binge_days DESC, user_id) AS rn
    FROM user_binge_counts
)
SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    user_id,
    binge_days
FROM top_user
WHERE rn = 1;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q7 — Project Question

**Answer: 1,250 Q1 signups.**  
**Never watched:** 226 Q1 signups.  
The query uses `LEFT JOIN` with `IS NULL`, avoiding the `NOT IN` + NULL trap.

In [ ]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(CASE WHEN ws.user_id IS NULL THEN 1 ELSE 0 END) AS never_watched
FROM users u
LEFT JOIN (
    SELECT DISTINCT user_id
    FROM watch_sessions
    WHERE user_id IS NOT NULL
) ws
    ON u.user_id = ws.user_id
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-03-31';
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q8 — Project Question

**Answer: 6 users** meet the Premium/Family current-plan + Basic-only watch-history condition.

In [ ]:
query = """
WITH current_active AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions s
    WHERE s.start_date <= '2024-06-30'
      AND (s.end_date IS NULL OR s.end_date > '2024-06-30')
      AND s.status = 'active'
),
premium_family_users AS (
    SELECT user_id
    FROM current_active
    WHERE rn = 1
      AND plan IN ('Premium', 'Family')
)
SELECT
    COUNT(*) AS overpaying_users
FROM premium_family_users p
WHERE EXISTS (
    SELECT 1
    FROM watch_sessions ws
    WHERE ws.user_id = p.user_id
)
AND NOT EXISTS (
    SELECT 1
    FROM watch_sessions ws
    JOIN shows sh
        ON ws.show_id = sh.show_id
    WHERE ws.user_id = p.user_id
      AND sh.min_plan IN ('Premium', 'Family')
);
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q9 — Project Question

**Answer: 55 users.**  
**Average days from signup to first upgrade:** 64.96 days.

In [ ]:
query = """
WITH ordered_subscriptions AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
),
first_subscription AS (
    SELECT
        user_id,
        plan AS first_plan,
        start_date AS first_start_date
    FROM ordered_subscriptions
    WHERE rn = 1
),
first_upgrade AS (
    SELECT
        o.user_id,
        MIN(o.start_date) AS first_upgrade_date
    FROM ordered_subscriptions o
    JOIN first_subscription f
        ON o.user_id = f.user_id
    WHERE f.first_plan = 'Basic'
      AND o.rn > 1
      AND o.plan IN ('Premium', 'Family')
    GROUP BY o.user_id
),
active_users AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE start_date <= '2024-06-30'
      AND (end_date IS NULL OR end_date > '2024-06-30')
      AND status = 'active'
)
SELECT
    COUNT(*) AS number_of_users,
    ROUND(
        AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)),
        2
    ) AS avg_days_signup_to_first_upgrade
FROM users u
JOIN first_subscription fs
    ON u.user_id = fs.user_id
JOIN first_upgrade fu
    ON u.user_id = fu.user_id
JOIN active_users au
    ON u.user_id = au.user_id
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-01-31';
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q10 — Project Question

**Answer: 4,345 cliffhanger comeback events.**  
**Top show:** S088 — Rayalaseema Raga, with 64 comeback events.

In [ ]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        a.user_id,
        a.show_id,
        a.session_date AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b
        ON a.user_id = b.user_id
       AND a.show_id = b.show_id
       AND b.session_date BETWEEN
           DATE_ADD(a.session_date, INTERVAL 1 DAY)
           AND DATE_ADD(a.session_date, INTERVAL 7 DAY)
    WHERE a.completed = 0
      AND a.user_id IS NOT NULL
),
show_counts AS (
    SELECT
        show_id,
        COUNT(*) AS comeback_count
    FROM comeback_events
    GROUP BY show_id
),
top_show AS (
    SELECT
        show_id,
        comeback_count,
        ROW_NUMBER() OVER (
            ORDER BY comeback_count DESC, show_id
        ) AS rn
    FROM show_counts
)
SELECT
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
    t.show_id,
    s.title,
    t.comeback_count
FROM top_show t
JOIN shows s
    ON t.show_id = s.show_id
WHERE t.rn = 1;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q11 — Project Question

**Answer: 1,675 users** had a streak of at least 4 consecutive weeks.  
**Longest streak:** 26 weeks.  
**One user with the longest streak:** U00213.

In [ ]:
query = """
WITH distinct_weeks AS (
    SELECT DISTINCT
        user_id,
        DATE_SUB(
            session_date,
            INTERVAL WEEKDAY(session_date) DAY
        ) AS week_start
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        week_start,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_start
        ) AS rn
    FROM distinct_weeks
),
streak_groups AS (
    SELECT
        user_id,
        week_start,
        DATE_SUB(week_start, INTERVAL rn WEEK) AS streak_group
    FROM numbered_weeks
),
streaks AS (
    SELECT
        user_id,
        streak_group,
        COUNT(*) AS streak_length
    FROM streak_groups
    GROUP BY user_id, streak_group
),
ranked_streaks AS (
    SELECT
        user_id,
        streak_length,
        ROW_NUMBER() OVER (
            ORDER BY streak_length DESC, user_id
        ) AS rn
    FROM streaks
)
SELECT
    (SELECT COUNT(DISTINCT user_id)
     FROM streaks
     WHERE streak_length >= 4) AS users_with_4plus_week_streak,
    (SELECT MAX(streak_length)
     FROM streaks) AS longest_streak_weeks,
    (SELECT user_id
     FROM ranked_streaks
     WHERE rn = 1) AS one_user_with_longest_streak;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Q12 — Project Question

**Answer: 521 churn-signal users.**  
These users had May 2024 watch activity and their June watch minutes dropped by at least 50%. The code prints every qualifying user with May minutes, June minutes, and drop percentage.

In [ ]:
query = """
WITH monthly_watch AS (
    SELECT
        user_id,
        SUM(
            CASE
                WHEN session_date BETWEEN '2024-05-01' AND '2024-05-31'
                THEN watch_minutes
                ELSE 0
            END
        ) AS may_watch_minutes,
        SUM(
            CASE
                WHEN session_date BETWEEN '2024-06-01' AND '2024-06-30'
                THEN watch_minutes
                ELSE 0
            END
        ) AS june_watch_minutes
    FROM watch_sessions
    WHERE session_date BETWEEN '2024-05-01' AND '2024-06-30'
      AND user_id IS NOT NULL
    GROUP BY user_id
),
churn_signals AS (
    SELECT
        user_id,
        may_watch_minutes,
        june_watch_minutes,
        ROUND(
            100.0 * (may_watch_minutes - june_watch_minutes)
            / may_watch_minutes,
            2
        ) AS drop_percentage
    FROM monthly_watch
    WHERE may_watch_minutes > 0
      AND june_watch_minutes <= may_watch_minutes * 0.50
)
SELECT
    u.user_id,
    u.name,
    cs.may_watch_minutes AS total_may_watch_minutes,
    cs.june_watch_minutes AS total_june_watch_minutes,
    cs.drop_percentage
FROM churn_signals cs
JOIN users u
    ON u.user_id = cs.user_id
ORDER BY u.user_id;
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

print("\nTotal churn signal users:", len(df))